In [24]:
import os, sys, glob, time
import traceback
from multiprocessing import Pool, cpu_count
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [25]:
from NEATObjects.NEAT import NEATTrainer
from GameObjects.Snake import SnakeGame
from ExperimentObjects.Experiment import Experiment

In [26]:
# --- Funkcja do uruchamiania pojedynczego eksperymentu ---
def run_experiment_for_config(
    config_file_path: str,
    base_results_dir: str,
    generations_count: int,
) -> tuple[str, float | None, str | None]:
    """
    Uruchamia eksperyment dla danego pliku konfiguracyjnego.
    config_file_path powinna być ścieżką absolutną.

    Zwraca:
        Tuple (nazwa_konfiguracji, wynik_fitness, ścieżka_do_stanów_gry_json)
    """
    config_name = os.path.splitext(os.path.basename(config_file_path))[0]
    print(f"Rozpoczynanie eksperymentu dla konfiguracji: {config_name}...")
    try:
        # Experiment.__init__ tworzy self.exp_dir = os.path.join(output_dir, cfg_name)
        # więc przekazujemy base_results_dir jako output_dir dla konstruktora Experiment.
        exp = Experiment(
            config_path=config_file_path,
            output_dir=base_results_dir,
            generations=generations_count,
        )
        final_fitness = exp.run()
        print(
            f"Zakończono eksperyment dla {config_name}. Końcowy fitness: {final_fitness:.2f}"
        )
        return config_name, final_fitness, exp.states_path
    except FileNotFoundError as fnfe:
        print(
            f"BŁĄD KRYTYCZNY (FileNotFoundError) podczas przetwarzania konfiguracji {config_name}: {fnfe}"
        )
        print(f"  Sprawdzana ścieżka do pliku konfiguracyjnego: {config_file_path}")
        print(
            f"  Jeśli błąd dotyczy pliku wewnątrz konfiguracji (np. initial_architecture),"
        )
        print(
            f"  sprawdź, czy ścieżka w pliku .ini jest poprawna względem lokalizacji pliku .ini lub projektu."
        )
        traceback.print_exc()
        return config_name, None, None
    except KeyError as ke:
        print(
            f"BŁĄD KRYTYCZNY (KeyError) podczas przetwarzania konfiguracji {config_name}: {ke}"
        )
        print(
            f"  Sprawdź, czy plik konfiguracyjny {config_file_path} zawiera wszystkie wymagane sekcje/klucze."
        )
        traceback.print_exc()
        return config_name, None, None
    except Exception as e:
        print(
            f"Inny błąd podczas przetwarzania konfiguracji {config_name}: {type(e).__name__} - {e}"
        )
        traceback.print_exc()
        return config_name, None, None

In [27]:
# --- Główna część skryptu ---
if __name__ == "__main__":
    # Ustawienia
    configs_folder_name = "Configs"  # Nazwa folderu z plikami .ini
    results_main_dir = "experiment_results_parallel"
    num_generations_per_exp = 225

    # Upewnij się, że główny katalog wyników istnieje
    # (Tworzymy go tutaj, bo może być potrzebny, zanim procesy potomne zaczną działać)
    os.makedirs(results_main_dir, exist_ok=True)

    # --- OKREŚLANIE ŚCIEŻEK ---
    # Pobierz ścieżkę do katalogu, w którym znajduje się TEN skrypt .py
    current_script_directory = os.path.dirname(os.path.abspath(__file__))

    # Zakładamy, że katalog główny projektu (NEAT_GAMING) to ten sam katalog,
    # w którym znajduje się ten skrypt.
    project_root_dir = current_script_directory
    # Jeśli ten skrypt byłby np. w NEAT_GAMING/Scripts/, to należałoby użyć:
    # project_root_dir = os.path.abspath(os.path.join(current_script_directory, ".."))

    absolute_configs_dir = os.path.join(project_root_dir, configs_folder_name)
    # --- KONIEC OKREŚLANIA ŚCIEŻEK ---

    # Znajdź wszystkie pliki konfiguracyjne.
    config_files_relative_or_absolute = glob.glob(
        os.path.join(absolute_configs_dir, "*.ini")
    )
    if not config_files_relative_or_absolute:
        print(
            f"Nie znaleziono plików konfiguracyjnych w folderze: {absolute_configs_dir}"
        )
        # Dodatkowy debug:
        print(f"  Sprawdzany katalog główny projektu: {project_root_dir}")
        print(f"  Bieżący katalog roboczy (CWD) podczas uruchamiania: {os.getcwd()}")
        sys.exit(1)

    # Używaj ścieżek absolutnych dla plików konfiguracyjnych
    config_files_list = [
        os.path.abspath(cfp) for cfp in config_files_relative_or_absolute
    ]

    print(
        f"Znaleziono {len(config_files_list)} plików konfiguracyjnych do przetworzenia."
    )
    for i, cfp in enumerate(config_files_list[:3]): # Pokaż pierwsze 3 dla weryfikacji
        print(f"  Przykład pliku konfiguracyjnego {i+1}: {cfp}")


    # Przygotuj argumenty dla każdego zadania
    tasks = [
        (cfp, results_main_dir, num_generations_per_exp)
        for cfp in config_files_list
    ]

    # Użyj tylu procesów, ile jest rdzeni CPU (lub mniej, jeśli jest mniej zadań)
    num_processes_to_use = min(len(config_files_list), cpu_count())
    print(
        f"Uruchamianie {len(tasks)} eksperymentów równolegle przy użyciu {num_processes_to_use} procesów..."
    )

    start_timestamp = time.time()

    # Uruchom eksperymenty równolegle
    with Pool(processes=num_processes_to_use) as pool:
        results_data = pool.starmap(run_experiment_for_config, tasks)

    end_timestamp = time.time()
    print(f"\n--- Wszystkie eksperymenty zakończone ---")
    print(
        f"Całkowity czas przetwarzania: {end_timestamp - start_timestamp:.2f} sekund."
    )

    # Wyświetl podsumowanie i informacje o odtwarzaniu
    print("\n--- Podsumowanie wyników ---")
    successful_exp_results = []
    for name, fitness_val, states_file_path in results_data:
        if fitness_val is not None:
            print(
                f"Konfiguracja: {name}, Końcowy Fitness: {fitness_val:.2f}, Stany gry: {states_file_path}"
            )
            successful_exp_results.append((name, fitness_val, states_file_path))
        else:
            print(f"Konfiguracja: {name} - nie powiodła się lub wystąpił błąd.")

    # Sortuj wyniki od najlepszego fitness
    if successful_exp_results: # Sortuj tylko jeśli są jakieś udane wyniki
        successful_exp_results.sort(key=lambda x: x[1], reverse=True)

    print("\n--- Najlepsze konfiguracje (Top 5) ---")
    if not successful_exp_results:
        print("Brak udanych eksperymentów do wyświetlenia.")
    else:
        for i, (name, fitness_val, _) in enumerate(
            successful_exp_results[:5]
        ):
            print(f"{i+1}. {name}: Fitness = {fitness_val:.2f}")

    # --- Odtwarzanie najlepszego wyniku ---
    if successful_exp_results:
        best_cfg_name, _, best_states_file = successful_exp_results[0]
        print(f"\n--- Odtwarzanie najlepszego eksperymentu ({best_cfg_name}) ---")

        original_config_file_for_replay = None
        for cfp_abs in config_files_list:
            if os.path.splitext(os.path.basename(cfp_abs))[0] == best_cfg_name:
                original_config_file_for_replay = cfp_abs
                break

        if original_config_file_for_replay:
            print(
                f"Oryginalny plik konfiguracyjny: {original_config_file_for_replay}"
            )
            print(f"Plik stanów gry: {best_states_file}")
            print(f"\nAby odtworzyć ręcznie (przykład):")
            print(
                f"  # Upewnij się, że Experiment jest poprawnie zaimportowany"
            )
            print(
                f"  exp_replayer = Experiment(config_path='{original_config_file_for_replay}', output_dir='{results_main_dir}')"
            )
            print(
                f"  # Załaduj stany, jeśli metoda replay tego wymaga lub użyj bezpośrednio:"
            )
            print(f"  if '{best_states_file}' and os.path.exists('{best_states_file}'):")
            print(
                f"      exp_replayer.game_play.replay('{best_states_file}', delay=0.05)"
            )
            print(f"  else: print('Plik stanów gry {best_states_file} nie istnieje lub jest pusty.')")

            try:
                print(
                    f"\nAutomatyczne odtwarzanie najlepszego wyniku dla: {best_cfg_name}..."
                )
                exp_replayer_auto = Experiment(
                    config_path=original_config_file_for_replay,
                    output_dir=results_main_dir,
                )
                if best_states_file and os.path.exists(best_states_file):
                    exp_replayer_auto.game_play.replay(best_states_file, delay=0.05)
                else:
                    print(
                        f"Nie można znaleźć pliku stanów gry do automatycznego odtworzenia: {best_states_file}"
                    )
            except Exception as e:
                print(f"Błąd podczas próby automatycznego odtworzenia: {e}")
                traceback.print_exc()
        else:
            print(
                f"Nie udało się znaleźć oryginalnego pliku .ini dla najlepszej konfiguracji '{best_cfg_name}' do odtworzenia."
            )
    else:
        print("\nBrak udanych eksperymentów, więc nie ma czego odtwarzać.")

    print("\nGotowe.")

NameError: name '__file__' is not defined

In [4]:
exp = Experiment('../Configs/my_custom_config.ini', output_dir='../Saved_outputs')
final_fitness = exp.run()


 ****** Running generation 0 ****** 
Population's average fitness: 0.78000 stdev: 1.10977
Best fitness: 7.00000 - size: (4, 400) - species 2 - id 37
Average adjusted fitness: 0.112
Mean genetic distance 2.861, standard deviation 0.569
Population of 200 members in 5 species:
   ID   age  size  fitness  adj fit  stag
  ====  ===  ====  =======  =======  ====
     1    0    45      5.0    0.112     0
     2    0     6      7.0    0.098     0
     3    0    41      4.0    0.126     0
     4    0    20       --       --     0
     5    0    88       --       --     0
Total extinctions: 0
Generation time: 14.847 sec

 ****** Running generation 1 ****** 
Population's average fitness: 0.75000 stdev: 1.03320
Best fitness: 5.00000 - size: (4, 396) - species 5 - id 267
Average adjusted fitness: 0.153
Mean genetic distance 2.851, standard deviation 0.610
Population of 200 members in 5 species:
   ID   age  size  fitness  adj fit  stag
  ====  ===  ====  =======  =======  ====
     1    1    23   

In [5]:
print(f"Final fitness: {final_fitness}")
exp.replay(delay=0.2)

Final fitness: 2.0
